In [10]:
# %%
import os
from pathlib import Path

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    GlobalAveragePooling2D
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [11]:
# %%
# =========================
# BASIC SETTINGS
# =========================

DATA_DIR = Path(r"D:\Exams\ICT\dataset")

IMG_HEIGHT = 224
IMG_WIDTH = 224
IMAGE_SIZE = (IMG_HEIGHT, IMG_WIDTH)

BATCH_SIZE = 32
TRAINING_EPOCHS = 15

SAVE_DIR = Path(r"D:\Exams\ICT\models")
SAVE_DIR.mkdir(exist_ok=True)

MODEL_PATH = SAVE_DIR / "efficientnetb0_sign_model.keras"

In [12]:
# %%
# =========================
# CLASS SELECTION
# =========================

classes_to_skip = {"J", "Z"}

sign_classes = [
    folder.name
    for folder in sorted(DATA_DIR.iterdir())
    if folder.is_dir() and folder.name.upper() not in classes_to_skip
]

TOTAL_CLASSES = len(sign_classes)

print("Classes selected:", sign_classes)
print("Total classes:", TOTAL_CLASSES)

Classes selected: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']
Total classes: 24


In [13]:
# %%
# =========================
# DATA AUGMENTATION
# =========================

image_data_generator = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.20,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True
)

In [14]:
# %%
# =========================
# CREATE DATA GENERATORS
# =========================

training_generator = image_data_generator.flow_from_directory(
    directory=str(DATA_DIR),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    classes=sign_classes,
    shuffle=True,
    seed=42
)

validation_generator = image_data_generator.flow_from_directory(
    directory=str(DATA_DIR),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    classes=sign_classes,
    shuffle=False,
    seed=42
)

print("Class indices:")
print(training_generator.class_indices)

Found 3458 images belonging to 24 classes.
Found 863 images belonging to 24 classes.
Class indices:
{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'K': 9, 'L': 10, 'M': 11, 'N': 12, 'O': 13, 'P': 14, 'Q': 15, 'R': 16, 'S': 17, 'T': 18, 'U': 19, 'V': 20, 'W': 21, 'X': 22, 'Y': 23}


In [15]:
# %%
# =========================
# EFFICIENTNETB0 MODEL
# =========================

model_input = Input(
    shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    name="input_image"
)

efficientnet_base = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_tensor=model_input
)

efficientnet_base.trainable = False

x = efficientnet_base.output
x = GlobalAveragePooling2D(name="average_pooling")(x)

x = Dense(
    units=256,
    activation="relu",
    name="classification_dense"
)(x)

x = Dropout(
    rate=0.5,
    name="regularization_dropout"
)(x)

model_output = Dense(
    units=TOTAL_CLASSES,
    activation="softmax",
    name="output_predictions"
)(x)

efficientnet_model = Model(
    inputs=model_input,
    outputs=model_output,
    name="efficientnetb0_transfer_model"
)

In [16]:
# %%
# =========================
# COMPILE MODEL
# =========================

efficientnet_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

efficientnet_model.summary()

Model: "efficientnetb0_transfer_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_image         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 224, 224,  │          0 │ input_image[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_1     │ (None, 224, 224,  │          7 │ rescaling_2[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_3         │ (None, 224, 224,  │          0 │ normalization_1[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_3[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,383,675 (16.72 MB)

 Trainable params: 334,104 (1.27 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [17]:
# %%
# =========================
# TRAIN MODEL
# =========================

efficientnet_history = efficientnet_model.fit(
    training_generator,
    validation_data=validation_generator,
    epochs=TRAINING_EPOCHS
)

Epoch 1/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 162s 1s/step - accuracy: 0.4881 - loss: 1.8000 - val_accuracy: 0.8888 - val_loss: 0.7109
Epoch 2/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 133s 1s/step - accuracy: 0.7718 - loss: 0.7503 - val_accuracy: 0.9397 - val_loss: 0.3798
Epoch 3/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 136s 1s/step - accuracy: 0.8548 - loss: 0.5010 - val_accuracy: 0.9363 - val_loss: 0.2690
Epoch 4/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 132s 1s/step - accuracy: 0.8875 - loss: 0.3819 - val_accuracy: 0.9594 - val_loss: 0.2088
Epoch 5/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 126s 1s/step - accuracy: 0.9080 - loss: 0.3157 - val_accuracy: 0.9803 - val_loss: 0.1363
Epoch 6/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 124s 1s/step - accuracy: 0.9193 - loss: 0.2699 - val_accuracy: 0.9768 - val_loss: 0.1320
Epoch 7/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 132s 1s/step - accuracy: 0.9286 - loss: 0.2500 - val_accuracy: 0.9722 - val_loss: 0.1107
Epoch 8/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 140s 1s/step - accuracy: 0.9430 - loss: 0.2067 - val_accu

In [18]:
# %%
# =========================
# SAVE TRAINED MODEL
# =========================

efficientnet_model.save(MODEL_PATH)

print(f"✅ EfficientNetB0 model saved at: {MODEL_PATH}")

✅ EfficientNetB0 model saved at: D:\Exams\ICT\models\efficientnetb0_sign_model.keras
